In [1]:
import os
import sys
import json
import random
import math
import time
from tqdm.auto import tqdm

from IPython.display import clear_output

import pandas as pd

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math, json, matplotlib
matplotlib.use("Agg")

import scipy
from scipy import io

import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn import functional as F
import torch.nn as nn
from torch.amp import autocast, GradScaler
from torchvision import transforms as T

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

# Dataset (halved size)

In [3]:
SLICE_MSS = 400
T_SIZE = int(80_000 / SLICE_MSS)

In [4]:
def normalize_times(info):
    new_start = 1000

    fixations_corrected = dict()
    fixations_corrected[new_start] = list(info['fixations'].items())[0][1]
    last_time = int(list(info['fixations'].items())[0][0].split('.')[0])
    current_time = new_start
    for f_time, f in info['fixations'].items():
        diff = min(1000, abs(int(f_time.split('.')[0]) - int(last_time)))
        last_time = int(f_time.split('.')[0])
        current_time = current_time + diff
        fixations_corrected[current_time] = f
    return fixations_corrected


def normalize_wordbounds(info):
    wordbounds_corrected = dict()
    for w_num, w_info in info['wordbounds'].items():
        w_dict = {
            "left": w_info['left'] / 2,
            "top":  w_info['top'] / 2,
            "right":  w_info['right'] / 2,
            "bottom":  w_info['bottom'] / 2,
            "is_read": 1
        }
        wordbounds_corrected[w_num] = w_dict
    return wordbounds_corrected


def stamp_gaussian(canvas: torch.Tensor, y: int, x: int, sigma: float = 1.5, amp: float = 1.0):
    H2, W2 = canvas.shape[-2], canvas.shape[-1]
    if x < 0 or y < 0 or x >= W2 or y >= H2:
        return
    r = int(math.ceil(3 * sigma))  # 3-sigma window
    y0, y1 = max(0, y - r), min(H2 - 1, y + r)
    x0, x1 = max(0, x - r), min(W2 - 1, x + r)
    yy = torch.arange(y0, y1 + 1, dtype=torch.float32) - y
    xx = torch.arange(x0, x1 + 1, dtype=torch.float32) - x
    gy = yy[:, None].pow(2)
    gx = xx[None, :].pow(2)
    kern = torch.exp(-(gy + gx) / (2.0 * sigma * sigma))
    patch = canvas[y0:y1 + 1, x0:x1 + 1].to(torch.float32)
    patch += amp * kern
    canvas[y0:y1 + 1, x0:x1 + 1] = patch.to(canvas.dtype)


def downsample_mask_preserve_gaps(mask_2d: torch.Tensor):
    x = mask_2d.unsqueeze(0).unsqueeze(0)  # (1,1,H,W)
    # min-pool: AND over 2x2 = 1 - max_pool(1 - x)
    y = 1.0 - F.max_pool2d(1.0 - x, kernel_size=2, stride=2)
    return y.squeeze(0).squeeze(0)

In [5]:
from collections.abc import Iterator


class AugmentedZucoTrainDataset(Dataset):
    def __init__(self, root_dir):
        self.root_dir = root_dir
        self.ids = sorted([
            (alpha, int(id))
            for alpha in os.listdir('data/train')
            for id in os.listdir(os.path.join('data/train', alpha))
        ])

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        alpha, id = self.ids[idx]
        path = os.path.join(self.root_dir, alpha, str(id))
        with open(os.path.join(path, 'info.json'), 'r') as j:
            info = json.loads(j.read())

        wordbounds = torch.load(os.path.join(path, 'wordbounds_tensor.pt')).to(torch.float32)
        content2 = downsample_mask_preserve_gaps(wordbounds.to(torch.float32))
        content2 = torch.ones(content2.shape)
        content2 = content2.to(torch.float16)

        fixations = normalize_times(info)
        wordbounds = normalize_wordbounds(info)

        # points scaled to halved grid (//2)
        moving_pts = torch.tensor(
            [[int(k / SLICE_MSS), int(v['x']) // 2, int(v['y']) // 2] for k, v in fixations.items()],
            dtype=torch.float16
        )
        fixed_pts_list = []
        for f_time, f in fixations.items():
            try:
                x_c = int(math.floor(f['x_corrected']))
                y_c = int(math.floor(f['y_corrected']))
            except:
                x_c = int(math.floor(f['x_before']))
                y_c = int(math.floor(f['y_before']))
            fixed_pts_list.append([int(f_time / SLICE_MSS), x_c // 2, y_c // 2])
        fixed_pts = torch.tensor(fixed_pts_list, dtype=torch.float16)

        # Build Gaussian fixation maps directly on the halved grid
        H2, W2 = 152, 104
        sigma_px = 1.5  # ~FWHM ≈ 2.355*sigma ≈ 3.5 px
        fix_frames = [torch.zeros((H2, W2), dtype=torch.float32) for _ in range(T_SIZE)]

        for t_ms, v in fixations.items():
            t_idx = int(t_ms / SLICE_MSS)
            if t_idx < 0 or t_idx >= T_SIZE:
                continue
            x2 = int(v['x']) // 2
            y2 = int(v['y']) // 2
            # amplitude normalized by 300ms (300ms -> 1.0). Clip to 1.0 (optional but sane).
            amp = float(v['duration']) / 300.0
            amp = min(1.0, max(0.0, amp))
            stamp_gaussian(fix_frames[t_idx], y2, x2, sigma=sigma_px, amp=amp)

        # Pack tensors for this sample
        fixations_3d = torch.stack([f.to(torch.float16) for f in fix_frames], dim=0) # (T,152,104)
        # content mask is static over time → repeat T times
        content_3d = content2.to(torch.float16).repeat(T_SIZE, 1, 1) # (T,152,104)
        page_3d = torch.stack([fixations_3d, content_3d], dim=0) # (2,T,152,104)

        return page_3d, moving_pts, fixed_pts, wordbounds


class AugmentedZucoTestDataset(Dataset):
    def __init__(self, root_dir):
        self.root_dir = root_dir
        self.ids = sorted([int(name) for name in os.listdir(root_dir)])

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        path = os.path.join(self.root_dir, str(self.ids[idx]))
        with open(os.path.join(path, 'info.json'), 'r') as j:
            info = json.loads(j.read())

        wordbounds = torch.load(os.path.join(path, 'wordbounds_tensor.pt')).to(torch.float32)
        content2 = downsample_mask_preserve_gaps(wordbounds.to(torch.float32))
        content2 = torch.ones(content2.shape)
        content2 = content2.to(torch.float16)

        fixations = normalize_times(info)
        wordbounds = normalize_wordbounds(info)

        moving_pts = torch.tensor(
            [[int(k / SLICE_MSS), int(v['x']) // 2, int(v['y']) // 2] for k, v in fixations.items()],
            dtype=torch.float16
        )
        fixed_pts_list = []
        for f_time, f in fixations.items():
            try:
                x_c = int(math.floor(f['x_corrected']))
                y_c = int(math.floor(f['y_corrected']))
            except:
                x_c = int(math.floor(f['x_before']))
                y_c = int(math.floor(f['y_before']))
            fixed_pts_list.append([int(f_time / SLICE_MSS), x_c // 2, y_c // 2])
        fixed_pts = torch.tensor(fixed_pts_list, dtype=torch.float16)

        H2, W2 = 152, 104
        sigma_px = 1.5
        fix_frames = [torch.zeros((H2, W2), dtype=torch.float32) for _ in range(T_SIZE)]

        for t_ms, v in fixations.items():
            t_idx = int(t_ms / SLICE_MSS)
            if t_idx < 0 or t_idx >= T_SIZE:
                continue
            x2 = int(v['x']) // 2
            y2 = int(v['y']) // 2
            amp = float(v['duration']) / 300.0
            amp = min(1.0, max(0.0, amp))
            stamp_gaussian(fix_frames[t_idx], y2, x2, sigma=sigma_px, amp=amp)

        fixations_3d = torch.stack([f.to(torch.float16) for f in fix_frames], dim=0)
        content_3d = content2.to(torch.float16).repeat(T_SIZE, 1, 1)
        page_3d = torch.stack([fixations_3d, content_3d], dim=0)
        return page_3d, moving_pts, fixed_pts, wordbounds


def collate_as_lists(batch):
    pages, movings, fixeds, _ = zip(*batch)
    pages = torch.stack(pages, dim=0)
    return pages, list(movings), list(fixeds)

In [6]:
train_val_data = AugmentedZucoTrainDataset('data/train')
train_size = int(len(train_val_data) * 0.90)
val_size = len(train_val_data) - train_size

train_data, val_data = torch.utils.data.random_split(train_val_data, [train_size, val_size], generator=torch.Generator().manual_seed(1))
test_data_0 = AugmentedZucoTestDataset('data/test/alpha_0')
test_data_750 = AugmentedZucoTestDataset('data/test/alpha_750')
test_data_1500 = AugmentedZucoTestDataset('data/test/alpha_1500')
test_data_3000 = AugmentedZucoTestDataset('data/test/alpha_3000')

# Model

In [7]:
def gn(c: int) -> nn.GroupNorm:
    # pick a divisor of c among {32,16,8,4,2,1}
    for g in (32, 16, 8, 4, 2, 1):
        if c % g == 0:
            return nn.GroupNorm(g, c)
    return nn.GroupNorm(1, c)


class ConvBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch, dilation=(1,1,1), norm_type: str = "gn"):
        super().__init__()
        pad = tuple(d for d in dilation)
        self.conv1 = nn.Conv3d(in_ch, out_ch, 3, padding=pad, dilation=dilation)
        self.conv2 = nn.Conv3d(out_ch, out_ch, 3, padding=pad, dilation=dilation)
        self.act = nn.LeakyReLU(0.1, inplace=True)
        if norm_type == "gn":
            self.n1 = gn(out_ch); self.n2 = gn(out_ch)
        elif norm_type == "none":
            self.n1 = nn.Identity(); self.n2 = nn.Identity()
        else:
            raise ValueError("norm_type must be 'gn' or 'none'")

    def forward(self, x):
        x = self.act(self.n1(self.conv1(x)))
        x = self.act(self.n2(self.conv2(x)))
        return x


class GazeMorph(nn.Module):
    def __init__(self, in_ch=2, base=32, max_disp=24.0):
        super().__init__()
        self.max_disp = max_disp

        # Encoder
        self.e1 = ConvBlock3D(in_ch, base,   norm_type="gn")   # (T,H,W)
        self.d1 = nn.Conv3d(base, base, 3, stride=(2,2,2), padding=1) # (T/2,H/2,W/2)
        self.e2 = ConvBlock3D(base, base*2, norm_type="gn")
        self.d2 = nn.Conv3d(base*2, base*2, 3, stride=(2,2,2), padding=1) # (T/4,H/4,W/4)
        self.e3 = ConvBlock3D(base*2, base*4, norm_type="gn")
        self.d3 = nn.Conv3d(base*4, base*4, 3, stride=(2,2,2), padding=1) # (T/8,H/8,W/8)
        self.e4 = ConvBlock3D(base*4, base*8, norm_type="gn")
        self.d4 = nn.Conv3d(base*8, base*8, 3, stride=(2,2,2), padding=1) # (T/16,H/16,W/16)

        self.b1 = ConvBlock3D(base*8, base*16, norm_type="gn")
        self.b2 = ConvBlock3D(base*16, base*16, norm_type="gn")

        # Decoder
        self.u4 = nn.ConvTranspose3d(base*16, base*8, kernel_size=2, stride=2, padding=1, output_padding=1)
        self.dec4 = ConvBlock3D(base*8 + base*8, base*8, norm_type="gn")
        
        self.u3 = nn.ConvTranspose3d(base*8, base*4,  kernel_size=(2,2,2), stride=(2,2,2)) # (T/4,H/4,W/4)
        self.dec3 = ConvBlock3D(base*4 + base*4, base*4, norm_type="gn")

        self.u2 = nn.ConvTranspose3d(base*4,  base*2,  kernel_size=(2,2,2), stride=(2,2,2)) # (T/2,H/2,W/2)
        self.dec2 = ConvBlock3D(base*2 + base*2, base*2, norm_type="none")

        self.u1 = nn.ConvTranspose3d(base*2, base,  kernel_size=(2,2,2), stride=(2,2,2)) # (T,H,W)
        self.dec1 = ConvBlock3D(base + base, base, norm_type="none")

        self.head = nn.Conv3d(base, 2, kernel_size=3, padding=1) # (dx,dy)

        for m in self.modules():
            if isinstance(m, (nn.Conv3d, nn.ConvTranspose3d)):
                nn.init.kaiming_normal_(m.weight, nonlinearity="leaky_relu")
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x):  # x: (B,2,T,H,W)
        x  = x.contiguous(memory_format=torch.channels_last_3d)
        e1 = self.e1(x)
        e2 = self.e2(self.d1(e1))
        e3 = self.e3(self.d2(e2))
        e4 = self.e4(self.d3(e3))

        b  = self.b2(self.b1(self.d4(e4)))

        d4 = self.u4(b); d4 = torch.cat([d4, e4], dim=1); d4 = self.dec4(d4)
        d3 = self.u3(d4); d3 = torch.cat([d3, e3], dim=1); d3 = self.dec3(d3)
        d2 = self.u2(d3); d2 = torch.cat([d2, e2], dim=1); d2 = self.dec2(d2)
        d1 = self.u1(d2); d1 = torch.cat([d1, e1], dim=1); d1 = self.dec1(d1)

        flow = self.head(d1)
        return torch.tanh(flow) * self.max_disp   # (B,2,T,H,W)

# Training

In [8]:
def _to_norm(coord, size, align_corners: bool):
    if size > 1:
        if align_corners:
            return 2.0 * coord / (size - 1.0) - 1.0
        else:
            return (2.0 * coord + 1.0) / size - 1.0
    else:
        return torch.zeros_like(coord)

# def warp_points_3d(pts_txy: torch.Tensor,
#                    warp_x: torch.Tensor,  # (T,H,W)
#                    warp_y: torch.Tensor,  # (T,H,W)
#                    align_corners: bool = True,
#                    padding_mode: str = "border",
#                    clamp_eps: float = 1e-6):
#     device = warp_x.device
#     dtype  = warp_x.dtype
#     pts = pts_txy.to(device=device, dtype=dtype)
#     T, H, W = warp_x.shape

#     # (1, 2, T, H, W)
#     field = torch.stack([warp_x, warp_y], dim=0).unsqueeze(0).to(dtype)

#     t = pts[:, 0]; x = pts[:, 1]; y = pts[:, 2]
#     gx = _to_norm(x, W, align_corners)
#     gy = _to_norm(y, H, align_corners)
#     gz = _to_norm(t, T, align_corners)

#     grid = torch.stack([gx, gy, gz], dim=-1).view(1, -1, 1, 1, 3)
#     if clamp_eps is not None:
#         grid = grid.clamp(-1 + clamp_eps, 1 - clamp_eps)

#     disp = F.grid_sample(field, grid, mode="bilinear",
#                          padding_mode=padding_mode,
#                          align_corners=align_corners)
#     # disp: (1, 2, N, 1, 1) -> (N, 2)
#     disp = disp[0, :, :, 0, 0].T

#     warped_pts = torch.stack([t, x + disp[:, 0], y + disp[:, 1]], dim=1)
#     return warped_pts, disp

# --- drop-in replacement: keep the same API you had before ---
def warp_points_3d(pts_txy: torch.Tensor,
                   warp_x: torch.Tensor,   # shape: (T,H,W)  or (B,T,H,W)
                   warp_y: torch.Tensor,   # shape: (T,H,W)  or (B,T,H,W)
                   clamp_to_bounds: bool = True):
    """
    Direct indexing + continuous update. No interpolation. No internal rounding.

    pts_txy: (N, 3) float tensor with columns (t, x, y) in *index units*.
             Semantics: these must already be integer indices (or extremely close).
    warp_x, warp_y: displacement fields. Unbatched: (T,H,W). Batched: (B,T,H,W).

    Returns:
      warped_pts: (N, 3) with (t, x+dx, y+dy)
      disp:       (N, 2) with (dx, dy) looked up exactly at those indices
    """
    # Normalize shapes to: (B,T,H,W)
    if warp_x.ndim == 3:
        warp_x = warp_x.unsqueeze(0)
        warp_y = warp_y.unsqueeze(0)
    assert warp_x.ndim == 4 and warp_y.ndim == 4, "warp_x/warp_y must be (B,T,H,W)"
    B, T, H, W = warp_x.shape

    pts = pts_txy.to(device=device, dtype=torch.float32)

    # We do NOT round. We only *verify* integrality (optional) and then cast for indexing.
    t_f, x_f, y_f = pts[:, 0], pts[:, 1], pts[:, 2]

    # After validation, cast to long for indexing
    t_idx = t_f.to(torch.long)
    x_idx = x_f.to(torch.long)
    y_idx = y_f.to(torch.long)

    if clamp_to_bounds:
        # No resizing: just safe guards
        t_idx = t_idx.clamp(0, T - 1)
        x_idx = x_idx.clamp(0, W - 1)
        y_idx = y_idx.clamp(0, H - 1)

    # If you run one session per batch, use b=0.
    # If your pipeline is batched with per-point batch ids, add a 'b_idx' column and index with it.
    b = 0
    dx = warp_x[b, t_idx, y_idx, x_idx]
    dy = warp_y[b, t_idx, y_idx, x_idx]

    warped_pts = torch.stack([pts[:, 0], pts[:, 1] + dx, pts[:, 2] + dy], dim=1)
    disp       = torch.stack([dx, dy], dim=1)
    return warped_pts, disp

def evaluate_point_dists(model, data_loader, device, amp_dtype, align_corners=True):
    model.eval()
    all_before, all_after = [], []
    with torch.no_grad(), autocast('cuda', enabled=(device.type=='cuda'), dtype=amp_dtype):
        for pages, mov_list, fix_list in data_loader:
            # keep dtype fp32; let autocast handle matmul/conv dtypes
            pages = pages.to(device, non_blocking=True).contiguous(memory_format=torch.channels_last_3d)
            flow = model(pages)  # (B,2,T,H,W)
            B = flow.shape[0]
            for b in range(B):
                mp = mov_list[b]; fp = fix_list[b]
                if mp.numel() == 0 or fp.numel() == 0: 
                    continue
                # warp landmarks with predicted flow (subpixel, differentiable sampling)
                warp_x = flow[b, 0]  # (T,H,W)
                warp_y = flow[b, 1]  # (T,H,W)
                warped_pts, _ = warp_points_3d(mp, warp_x, warp_y)

                before = (mp[:, 1:3].to(torch.float32) - fp[:, 1:3].to(torch.float32)).norm(dim=1).cpu()
                after = (warped_pts[:, 1:3].to(torch.float32).cpu() - fp[:, 1:3].to(torch.float32)).norm(dim=1).cpu()
                all_before.append(before)
                all_after.append(after)
    if len(all_before) == 0:
        return 0.0, 0.0, torch.empty(0), torch.empty(0)
    d_before = torch.cat(all_before, dim=0)
    d_after  = torch.cat(all_after,  dim=0)
    return d_before.mean().item(), d_after.mean().item(), d_before, d_after

def evaluate_point_dists_non_batch(model, data_loader, device, amp_dtype, align_corners=True):
    model.eval()
    all_before, all_after = [], []
    with torch.no_grad(), autocast('cuda', enabled=(device.type=='cuda'), dtype=amp_dtype):
        for pages, mov_list, fix_list, _ in data_loader:
            # keep dtype fp32; let autocast handle matmul/conv dtypes
            pages = pages.to(device)
            flow = model(pages.unsqueeze(0))  # (2,T,H,W)
            flow = flow.squeeze(0)
            mp = mov_list; fp = fix_list
            if mp.numel() == 0 or fp.numel() == 0: 
                continue
            # warp landmarks with predicted flow (subpixel, differentiable sampling)
            warp_x = flow[0]  # (T,H,W)
            warp_y = flow[1]  # (T,H,W)
            warped_pts, _ = warp_points_3d(mp, warp_x, warp_y)

            before = (mp[:, 1:3].to(torch.float32) - fp[:, 1:3].to(torch.float32)).norm(dim=1).cpu()
            after = (warped_pts[:, 1:3].to(torch.float32).cpu() - fp[:, 1:3].to(torch.float32)).norm(dim=1).cpu()
            all_before.append(before)
            all_after.append(after)
    if len(all_before) == 0:
        return 0.0, 0.0, torch.empty(0), torch.empty(0)
    d_before = torch.cat(all_before, dim=0)
    d_after  = torch.cat(all_after,  dim=0)
    return d_before.mean().item(), d_after.mean().item(), d_before, d_after

def save_dists_hist_jpg(d_before, d_after, out_path, title="Distances (px) before vs after"):
    d_before = d_before.numpy() if torch.is_tensor(d_before) else d_before
    d_after  = d_after.numpy()  if torch.is_tensor(d_after)  else d_after
    plt.figure(figsize=(7,5), dpi=150)
    bins = 10
    plt.hist(d_before, bins=bins, alpha=0.5, label="before", density=True)
    plt.hist(d_after,  bins=bins, alpha=0.5, label="after",  density=True)
    plt.xlabel("Euclidean distance (pixels)")
    plt.ylabel("Density")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path)
    plt.close()

In [9]:
def spatial_smoothness(flow):
    # membrane energy on each time-slice
    dx = flow[:, :, :, :, 1:] - flow[:, :, :, :, :-1]
    dy = flow[:, :, :, 1:, :] - flow[:, :, :, :-1, :]
    return 0.5 * (dx.square().mean() + dy.square().mean())

def temporal_smoothness(flow):
    dt = flow[:, :, 1:, :, :] - flow[:, :, :-1, :, :]
    return dt.square().mean()

def jacobian_fold_penalty(flow):
    # penalize negative det(J) of (x+u, y+v)
    u = flow[:, 0:1]; v = flow[:, 1:2]
    ux = u[:, :, :, :, 1:] - u[:, :, :, :, :-1]
    uy = u[:, :, :, 1:, :] - u[:, :, :, :-1, :]
    vx = v[:, :, :, :, 1:] - v[:, :, :, :, :-1]
    vy = v[:, :, :, 1:, :] - v[:, :, :, :-1, :]
    Hm = min(ux.shape[-2], uy.shape[-2], vx.shape[-2], vy.shape[-2])
    Wm = min(ux.shape[-1], uy.shape[-1], vx.shape[-1], vy.shape[-1])
    ux = ux[..., :Hm, :Wm]; uy = uy[..., :Hm, :Wm]
    vx = vx[..., :Hm, :Wm]; vy = vy[..., :Hm, :Wm]
    detJ = (1 + ux) * (1 + vy) - (uy * vx)
    return F.relu(-detJ).mean()

def point_correspondence_loss(flow, moving_pts_list, fixed_pts_list):
    # lists of (Li,3) with (t,x,y) on the halved grid
    B, _, T, H, W = flow.shape
    total = flow.new_tensor(0.); count = 0
    for b in range(B):
        mp = moving_pts_list[b]; fp = fixed_pts_list[b]
        if mp.numel() == 0 or fp.numel() == 0: continue
        L = min(mp.shape[0], fp.shape[0])
        mp = mp[:L].to(flow.device, dtype=torch.float32)
        fp = fp[:L].to(flow.device, dtype=torch.float32)
        t = mp[:, 0].long().clamp_(0, T-1)
        x = mp[:, 1].long().clamp_(0, W-1)
        y = mp[:, 2].long().clamp_(0, H-1)
        dx = flow[b, 0, t, y, x]; dy = flow[b, 1, t, y, x]
        pred_x = mp[:, 1] + dx;  pred_y = mp[:, 2] + dy
        total += F.smooth_l1_loss(pred_x, fp[:, 1], reduction='mean') \
               + F.smooth_l1_loss(pred_y, fp[:, 2], reduction='mean')
        count += 1
    return total if count == 0 else total / count

In [10]:
def _gpu_mem_gb():
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / (1024**3)
        reserv = torch.cuda.memory_reserved() / (1024**3)
        return f"{alloc:.2f}/{reserv:.2f} GB"
    return "CPU"

def train_gmorph(
       model, epochs=5, lr=1e-4,
       lambda_spatial=0.035, lambda_time=0.05, lambda_fold=0.10,
       batch_size=4, num_workers=4,
       amp_dtype=(torch.bfloat16 if (device.type=="cuda" and torch.cuda.is_bf16_supported()) else torch.float16),
       warmup_epochs=1, min_lr=1e-6, weight_decay=1e-6,
       eval_every=5,               # evaluate dists & write metrics every K epochs
       save_every=1,               # save a rolling epoch checkpoint every K epochs
       resume_ckpt=None,           # path to resume from
       ckpt_prefix="gmorph",       # filename prefix for saved checkpoints
):
    train_loader = torch.utils.data.DataLoader(
        train_data, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=True,
        persistent_workers=(num_workers > 0), collate_fn=collate_as_lists,
    )
    val_loader = torch.utils.data.DataLoader(
        val_data, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True,
        persistent_workers=(num_workers > 0), collate_fn=collate_as_lists,
    )

    optim = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scaler = GradScaler('cuda', enabled=(device.type=='cuda' and amp_dtype==torch.float16))
    # warmup_epochs = max(0, int(warmup_epochs))
    # main_epochs = max(1, epochs - warmup_epochs)
    # scheds = []
    # if warmup_epochs > 0:
    #     scheds.append(torch.optim.lr_scheduler.LinearLR(optim, start_factor=0.1, total_iters=warmup_epochs))
    # scheds.append(torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=main_epochs, eta_min=min_lr))
    # scheduler = (torch.optim.lr_scheduler.SequentialLR(optim, scheds, milestones=[warmup_epochs]) if len(scheds) > 1 else scheds[0])

    start_epoch = 1
    best_val = float('inf')
    if resume_ckpt is not None and os.path.isfile(resume_ckpt):
        ckpt = torch.load(resume_ckpt, map_location="cpu")
        model.load_state_dict(ckpt["model"])
        optim.load_state_dict(ckpt["optim"])
        if ckpt.get("scaler") is not None and scaler.is_enabled():
            scaler.load_state_dict(ckpt["scaler"])
        if ckpt.get("scheduler") is not None:
            scheduler.load_state_dict(ckpt["scheduler"])
        start_epoch = int(ckpt.get("epoch", 0)) + 1
        best_val = float(ckpt.get("best_val", float("inf")))

    # if warmup_epochs > 0:
    #     for pg in optim.param_groups:
    #         pg['lr'] = lr * 0.1

    history = {
        "epoch": [], "lr": [],
        "train_loss": [], "val_loss": [],
        "train_lp": [], "train_ls": [], "train_lt": [], "train_lf": [],
        "val_lp": [],   "val_ls": [],   "val_lt": [],   "val_lf": [],
    }

    for epoch in range(start_epoch, epochs+1):
        model.train()
        running = 0.0; sum_lp = 0.0; sum_ls = 0.0; sum_lt = 0.0; sum_lf = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs} [train]", leave=False)
        for pages, mov_list, fix_list in pbar:
            pages = pages.to(device, non_blocking=True).contiguous(memory_format=torch.channels_last_3d)
            optim.zero_grad(set_to_none=True)

            with autocast('cuda', enabled=(device.type=='cuda'), dtype=amp_dtype):
                flow = model(pages)

                lp = point_correspondence_loss(flow, mov_list, fix_list)
                ls = spatial_smoothness(flow)
                lt = temporal_smoothness(flow)
                lf = jacobian_fold_penalty(flow)

                loss = lp + lambda_spatial*ls + lambda_time*lt + lambda_fold*lf

            if scaler.is_enabled():
                scaler.scale(loss).backward()
                scaler.unscale_(optim)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                scaler.step(optim); scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optim.step()

            running += loss.item()
            sum_lp += lp.item(); sum_ls += ls.item(); sum_lt += lt.item(); sum_lf += lf.item()
            pbar.set_postfix({
                "loss": f"{loss.item():.3f}",
                "lp": f"{lp.item():.3f}",
                "ls": f"{ls.item():.3f}",
                "lt": f"{lt.item():.3f}",
                "lf": f"{lf.item():.3f}",
                "mem": _gpu_mem_gb(),
            })

        train_loss = running / max(1, len(train_loader))
        train_lp = sum_lp / max(1, len(train_loader))
        train_ls = sum_ls / max(1, len(train_loader))
        train_lt = sum_lt / max(1, len(train_loader))
        train_lf = sum_lf / max(1, len(train_loader))

        model.eval()
        running = 0.0; sum_lp = 0.0; sum_ls = 0.0; sum_lt = 0.0; sum_lf = 0.0
        pbarv = tqdm(val_loader, desc=f"Epoch {epoch}/{epochs} [val]  ", leave=False)
        with torch.no_grad(), autocast('cuda', enabled=(device.type=='cuda'), dtype=amp_dtype):
            for pages, mov_list, fix_list in pbarv:
                pages = pages.to(device, non_blocking=True).contiguous(memory_format=torch.channels_last_3d)
                flow = model(pages)

                lp = point_correspondence_loss(flow, mov_list, fix_list)
                ls = spatial_smoothness(flow)
                lt = temporal_smoothness(flow)
                lf = jacobian_fold_penalty(flow)

                loss = lp + lambda_spatial*ls + lambda_time*lt + lambda_fold*lf
                running += loss.item()
                sum_lp += lp.item(); sum_ls += ls.item(); sum_lt += lt.item(); sum_lf += lf.item()

                pbarv.set_postfix({
                    "loss": f"{loss.item():.3f}",
                    "lp": f"{lp.item():.3f}",
                    "ls": f"{ls.item():.3f}",
                    "lt": f"{lt.item():.3f}",
                    "lf": f"{lf.item():.3f}",
                    "mem": _gpu_mem_gb(),
                })

        val_loss = running / max(1, len(val_loader))
        val_lp = sum_lp / max(1, len(val_loader))
        val_ls = sum_ls / max(1, len(val_loader))
        val_lt = sum_lt / max(1, len(val_loader))
        val_lf = sum_lf / max(1, len(val_loader))

        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), os.path.join(MODEL_DIR, "best_state_dict.tr"))
        if (epoch % save_every) == 0:
            ckpt_path = os.path.join(MODEL_DIR, f"{ckpt_prefix}_epoch{epoch:04d}.pt")
            torch.save(
                {
                    "model": model.state_dict(),
                    "optim": optim.state_dict(),
                    "scaler": (scaler.state_dict() if scaler.is_enabled() else None),
                    # "scheduler": scheduler.state_dict(),
                    "epoch": epoch,
                    "best_val": best_val,
                },
                ckpt_path,
            )

        # lr_now = optim.param_groups[0]["lr"]
        print(
            f"[Epoch {epoch}/{epochs}] "
            f"train={train_loss:.4f}  val={val_loss:.4f}  best={best_val:.4f}"
            f"mem={_gpu_mem_gb()}"
        )

        history["epoch"].append(epoch)
        # history["lr"].append(lr_now)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_lp"].append(train_lp); history["train_ls"].append(train_ls)
        history["train_lt"].append(train_lt); history["train_lf"].append(train_lf)
        history["val_lp"].append(val_lp);     history["val_ls"].append(val_ls)
        history["val_lt"].append(val_lt);     history["val_lf"].append(val_lf)

        # scheduler.step()

        # # periodic eval of landmark dists and logging
        if (epoch % eval_every) == 0 or epoch == epochs:
            mean_before_val, mean_after_val, d_before_val, d_after_val = evaluate_point_dists(
                model, val_loader, device=device, amp_dtype=amp_dtype
            )
            mean_before_0, mean_after_0, d_before_0, d_after_0 = evaluate_point_dists_non_batch(
                model, test_data_0, device=device, amp_dtype=amp_dtype
            )
            mean_before_750, mean_after_750, d_before_750, d_after_750 = evaluate_point_dists_non_batch(
                model, test_data_750, device=device, amp_dtype=amp_dtype
            )
            mean_before_1500, mean_after_1500, d_before_1500, d_after_1500 = evaluate_point_dists_non_batch(
                model, test_data_1500, device=device, amp_dtype=amp_dtype
            )
            mean_before_3000, mean_after_3000, d_before_3000, d_after_3000 = evaluate_point_dists_non_batch(
                model, test_data_3000, device=device, amp_dtype=amp_dtype
            )
            # histogram per eval checkpoint
            jpg_path = os.path.join(MODEL_DIR, f'dists_before_after_epoch_{epoch}.jpg')
            base, ext = os.path.splitext(jpg_path)
            save_dists_hist_jpg(d_before_val, d_after_val, f"{base}_e{epoch:04d}_val{ext}",
                                title=f"Dists before vs after @ epoch {epoch} @ val")
            save_dists_hist_jpg(d_before_0, d_after_0, f"{base}_e{epoch:04d}_0{ext}",
                                title=f"Dists before vs after @ epoch {epoch} @ 0")
            save_dists_hist_jpg(d_before_750, d_after_750, f"{base}_e{epoch:04d}_750{ext}",
                                title=f"Dists before vs after @ epoch {epoch} @ 750")
            save_dists_hist_jpg(d_before_1500, d_after_1500, f"{base}_e{epoch:04d}_1500{ext}",
                                title=f"Dists before vs after @ epoch {epoch} @ 1500")
            save_dists_hist_jpg(d_before_3000, d_after_3000, f"{base}_e{epoch:04d}_3000{ext}",
                                title=f"Dists before vs after @ epoch {epoch} @ 3000")
            # append metrics (json-per-line) to txt
            line = {
                "epoch": epoch,
                # "lr": optim.param_groups[0]["lr"],
                "train_loss": train_loss, "val_loss": val_loss, "best_val": best_val,
                "train_lp": train_lp, "train_ls": train_ls, "train_lt": train_lt, "train_lf": train_lf,
                "val_lp": val_lp, "val_ls": val_ls, "val_lt": val_lt, "val_lf": val_lf,
                "mean_dist_before_val": mean_before_val, "mean_dist_after_val": mean_after_val,
                "mean_dist_before_0": mean_before_0, "mean_dist_after_0": mean_after_0,
                "mean_dist_before_750": mean_before_750, "mean_dist_after_750": mean_after_750,
                "mean_dist_before_1500": mean_before_1500, "mean_dist_after_1500": mean_after_1500,
                "mean_dist_before_3000": mean_before_3000, "mean_dist_after_3000": mean_after_3000,
            }
            out_txt = os.path.join(MODEL_DIR, f'metrics_epoch_{epoch}.txt')
            with open(out_txt, "a", encoding="utf-8") as f:
                f.write(json.dumps(line) + "\n")
    metrics = {
        "epochs": epochs,
        "best_val": best_val,
        "history": history,
    }
    return model, metrics

In [11]:
MODEL_DIR = 'GazeMorphV3_no_content_16_25_disp24_halving_on_stride'

model = GazeMorph(in_ch=2, base=16, max_disp=24).to(device)
amp_dtype = (torch.bfloat16 if (device.type=="cuda" and torch.cuda.is_bf16_supported()) else torch.float16)

model, metrics = train_gmorph(
    model=model,
    epochs=20,
    lr=1e-4,
    weight_decay=1e-5,
    lambda_spatial=0.035,
    lambda_time=0.05,
    lambda_fold=0.10,
    batch_size=4,
    num_workers=4,
    amp_dtype=amp_dtype,
    # warmup_epochs=1,
    # min_lr=2e-5,
    eval_every=5,
    save_every=5,
    ckpt_prefix="gmorph",
    # resume_ckpt='GazeMorphV3_32_25_disp24_halving_on_stride/gmorph_epoch0010.pt'
)

Epoch 1/20 [train]:   0%|          | 0/3600 [00:00<?, ?it/s]

Epoch 1/20 [val]  :   0%|          | 0/400 [00:00<?, ?it/s]

[Epoch 1/20] train=4.4622  val=3.8113  best=3.8113mem=0.24/14.10 GB


Epoch 2/20 [train]:   0%|          | 0/3600 [00:00<?, ?it/s]

Epoch 2/20 [val]  :   0%|          | 0/400 [00:00<?, ?it/s]

...

Epoch 19/20 [val]  :   0%|          | 0/400 [00:00<?, ?it/s]

[Epoch 19/20] train=2.5065  val=3.0058  best=2.9664mem=0.24/14.11 GB


Epoch 20/20 [train]:   0%|          | 0/3600 [00:00<?, ?it/s]

Epoch 20/20 [val]  :   0%|          | 0/400 [00:00<?, ?it/s]

[Epoch 20/20] train=2.4594  val=2.9415  best=2.9415mem=0.24/14.11 GB


# Metrics

In [11]:
def _to_norm(coord, size, align_corners: bool):
    if size > 1:
        if align_corners:
            return 2.0 * coord / (size - 1.0) - 1.0
        else:
            return (2.0 * coord + 1.0) / size - 1.0
    else:
        return torch.zeros_like(coord)

# def warp_points_3d(pts_txy: torch.Tensor,
#                    warp_x: torch.Tensor,  # (T,H,W)
#                    warp_y: torch.Tensor,  # (T,H,W)
#                    align_corners: bool = True,
#                    padding_mode: str = "border",
#                    clamp_eps: float = 1e-6):
#     device = warp_x.device
#     dtype  = warp_x.dtype
#     pts = pts_txy.to(device=device, dtype=dtype)
#     T, H, W = warp_x.shape

#     # (1, 2, T, H, W)
#     field = torch.stack([warp_x, warp_y], dim=0).unsqueeze(0).to(dtype)

#     t = pts[:, 0]; x = pts[:, 1]; y = pts[:, 2]
#     gx = _to_norm(x, W, align_corners)
#     gy = _to_norm(y, H, align_corners)
#     gz = _to_norm(t, T, align_corners)

#     grid = torch.stack([gx, gy, gz], dim=-1).view(1, -1, 1, 1, 3)
#     if clamp_eps is not None:
#         grid = grid.clamp(-1 + clamp_eps, 1 - clamp_eps)

#     disp = F.grid_sample(field, grid, mode="bilinear",
#                          padding_mode=padding_mode,
#                          align_corners=align_corners)
#     # disp: (1, 2, N, 1, 1) -> (N, 2)
#     disp = disp[0, :, :, 0, 0].T

#     warped_pts = torch.stack([t, x + disp[:, 0], y + disp[:, 1]], dim=1)
#     return warped_pts, disp

In [12]:
MODEL_DIR = 'GazeMorphV3_16_25_disp24_halving_on_stride/gmorph_epoch0020.pt'

model = GazeMorph(in_ch=2, base=16, max_disp=24.0).to(device)
state = torch.load(MODEL_DIR, map_location=device)

ckpt = torch.load(MODEL_DIR, map_location="cpu")
model.load_state_dict(ckpt["model"])

<All keys matched successfully>

## Mean distance

In [18]:
N = 0
before_sum = 0
after_sum = 0

for page, mov_list, fix_list, _ in tqdm(test_data_0):
    model.eval()
    with torch.no_grad(), autocast('cuda', enabled=(device.type=='cuda'), dtype=(torch.bfloat16 if (device.type=="cuda" and torch.cuda.is_bf16_supported()) else torch.float16)):
        page = page.to(device, dtype=torch.float16, non_blocking=True).unsqueeze(0)
        flow = model(page).squeeze(0)
    warped_pts = warp_points_3d(mov_list, flow[0], flow[1])[0]

    dists_before = (mov_list[:, 1:3].to(torch.float32) - fix_list[:, 1:3].to(torch.float32)).norm(dim=1)
    dists_after = (warped_pts[:, 1:3].to(torch.float32).to(torch.device('cpu')) - fix_list[:, 1:3].to(torch.float32)).norm(dim=1)

    N += mov_list.shape[0]
    before_sum += dists_before.sum()
    after_sum += dists_after.sum()

print(before_sum / N, after_sum / N)

  0%|          | 0/550 [00:00<?, ?it/s]

tensor(1.5395) tensor(14.8758)


In [12]:
N = 0
before_sum = 0
after_sum = 0

for page, mov_list, fix_list, _ in tqdm(test_data_750):
    model.eval()
    with torch.no_grad(), autocast('cuda', enabled=(device.type=='cuda'), dtype=(torch.bfloat16 if (device.type=="cuda" and torch.cuda.is_bf16_supported()) else torch.float16)):
        page = page.to(device, dtype=torch.float16, non_blocking=True).unsqueeze(0)
        flow = model(page).squeeze(0)
    warped_pts = warp_points_3d(mov_list, flow[0], flow[1])[0]

    dists_before = (mov_list[:, 1:3].to(torch.float32) - fix_list[:, 1:3].to(torch.float32)).norm(dim=1)
    dists_after = (warped_pts[:, 1:3].to(torch.float32).to(torch.device('cpu')) - fix_list[:, 1:3].to(torch.float32)).norm(dim=1)

    N += mov_list.shape[0]
    before_sum += dists_before.sum()
    after_sum += dists_after.sum()

print(before_sum / N, after_sum / N)

  0%|          | 0/550 [00:00<?, ?it/s]

tensor(2.7027) tensor(1.9696)


In [13]:
N = 0
before_sum = 0
after_sum = 0

for page, mov_list, fix_list, _ in tqdm(test_data_1500):
    model.eval()
    with torch.no_grad(), autocast('cuda', enabled=(device.type=='cuda'), dtype=(torch.bfloat16 if (device.type=="cuda" and torch.cuda.is_bf16_supported()) else torch.float16)):
        page = page.to(device, dtype=torch.float16, non_blocking=True).unsqueeze(0)
        flow = model(page).squeeze(0)
    warped_pts = warp_points_3d(mov_list, flow[0], flow[1])[0]

    dists_before = (mov_list[:, 1:3].to(torch.float32) - fix_list[:, 1:3].to(torch.float32)).norm(dim=1)
    dists_after = (warped_pts[:, 1:3].to(torch.float32).to(torch.device('cpu')) - fix_list[:, 1:3].to(torch.float32)).norm(dim=1)

    N += mov_list.shape[0]
    before_sum += dists_before.sum()
    after_sum += dists_after.sum()

print(before_sum / N, after_sum / N)

  0%|          | 0/550 [00:00<?, ?it/s]

tensor(4.5358) tensor(2.8348)


In [13]:
N = 0
before_sum = 0
after_sum = 0

for page, mov_list, fix_list, _ in tqdm(test_data_3000):
    model.eval()
    with torch.no_grad(), autocast('cuda', enabled=(device.type=='cuda'), dtype=(torch.bfloat16 if (device.type=="cuda" and torch.cuda.is_bf16_supported()) else torch.float16)):
        page = page.to(device, dtype=torch.float16, non_blocking=True).unsqueeze(0)
        flow = model(page).squeeze(0)
    warped_pts = warp_points_3d(mov_list, flow[0], flow[1])[0]

    dists_before = (mov_list[:, 1:3].to(torch.float32) - fix_list[:, 1:3].to(torch.float32)).norm(dim=1)
    dists_after = (warped_pts[:, 1:3].to(torch.float32).to(torch.device('cpu')) - fix_list[:, 1:3].to(torch.float32)).norm(dim=1)

    N += mov_list.shape[0]
    before_sum += dists_before.sum()
    after_sum += dists_after.sum()

print(before_sum / N, after_sum / N)

  0%|          | 0/550 [00:00<?, ?it/s]

tensor(8.5594) tensor(4.8760)


## All read (mIoU)

In [13]:
def mIoU_stats(data, model):
    N = 0
    IoU_sum = 0
    for page, mov_list, fix_list, wordbounds in tqdm(data):
        model.eval()
        with torch.no_grad(), autocast('cuda', enabled=(device.type=='cuda'), dtype=(torch.bfloat16 if (device.type=="cuda" and torch.cuda.is_bf16_supported()) else torch.float16)):
            page = page.to(device, dtype=torch.float16, non_blocking=True).unsqueeze(0)
            flow = model(page).squeeze(0)
        warped_pts = warp_points_3d(mov_list, flow[0], flow[1])[0]
        # warped_pts = mov_list

        true_read_words = set()
        for _, fx, fy in fix_list:
            nearest_word_num = None
            min_distance = 9999999999
            for w_num, w in wordbounds.items():
                right = int(math.floor(w['right']))
                bottom = int(math.floor(w['bottom']))
                left = int(math.floor(w['left']))
                top = int(math.floor(w['top']))
                dists = [math.sqrt((fx-left)**2+(fy-top)**2), math.sqrt((fx-right)**2+(fy-top)**2), math.sqrt((fx-right)**2+(fy-bottom)**2), math.sqrt((fx-left)**2+(fy-bottom)**2)]
                if fx < right and fx > left:
                    dists.append(abs(fy-top))
                    dists.append(abs(fy-bottom))
                if fy < bottom and fy > top:
                    dists.append(abs(fx-right))
                    dists.append(abs(fx-left))
                min_d = min(dists)
                if min_d < min_distance:
                    nearest_word_num = int(w_num)
                    min_distance = min_d
            true_read_words.add(int(nearest_word_num))

        pred_read_words = set()
        for _, fx, fy in warped_pts:
            nearest_word_num = None
            min_distance = 9999999999
            for w_num, w in wordbounds.items():
                right = int(math.floor(w['right']))
                bottom = int(math.floor(w['bottom']))
                left = int(math.floor(w['left']))
                top = int(math.floor(w['top']))
                dists = [math.sqrt((fx-left)**2+(fy-top)**2), math.sqrt((fx-right)**2+(fy-top)**2), math.sqrt((fx-right)**2+(fy-bottom)**2), math.sqrt((fx-left)**2+(fy-bottom)**2)]
                if fx < right and fx > left:
                    dists.append(abs(fy-top))
                    dists.append(abs(fy-bottom))
                if fy < bottom and fy > top:
                    dists.append(abs(fx-right))
                    dists.append(abs(fx-left))
                min_d = min(dists)
                if min_d < min_distance:
                    nearest_word_num = int(w_num)
                    min_distance = min_d
            pred_read_words.add(int(nearest_word_num))
        IoU = len(true_read_words.intersection(pred_read_words)) / len(true_read_words.union(pred_read_words))
        IoU_sum += IoU
        N += 1
    return IoU_sum / N


def mIoU_stats_old(data, model):
    N = 0
    IoU_sum = 0
    for page, mov_list, fix_list, wordbounds in tqdm(data):
        # model.eval()
        # with torch.no_grad(), autocast('cuda', enabled=(device.type=='cuda'), dtype=(torch.bfloat16 if (device.type=="cuda" and torch.cuda.is_bf16_supported()) else torch.float16)):
        #     page = page.to(device, dtype=torch.float16, non_blocking=True).unsqueeze(0)
        #     flow = model(page).squeeze(0)
        # warped_pts = warp_points_3d(mov_list, flow[0], flow[1])[0]
        warped_pts = mov_list

        true_read_words = set()
        for _, fx, fy in fix_list:
            for w_num, w in wordbounds.items():
                right = int(math.floor(w['right']))
                bottom = int(math.floor(w['bottom']))
                left = int(math.floor(w['left']))
                top = int(math.floor(w['top']))
                if fx < right and fx > left and fy < bottom and fy > top:
                    true_read_words.add(int(w_num))

        pred_read_words = set()
        for _, fx, fy in warped_pts:
            nearest_word_num = None
            min_distance = 9999999999
            for w_num, w in wordbounds.items():
                right = int(math.floor(w['right']))
                bottom = int(math.floor(w['bottom']))
                left = int(math.floor(w['left']))
                top = int(math.floor(w['top']))
                dists = [math.sqrt((fx-left)**2+(fy-top)**2), math.sqrt((fx-right)**2+(fy-top)**2), math.sqrt((fx-right)**2+(fy-bottom)**2), math.sqrt((fx-left)**2+(fy-bottom)**2)]
                if fx < right and fx > left:
                    dists.append(abs(fy-top))
                    dists.append(abs(fy-bottom))
                if fy < bottom and fy > top:
                    dists.append(abs(fx-right))
                    dists.append(abs(fx-left))
                min_d = min(dists)
                if min_d < min_distance:
                    nearest_word_num = int(w_num)
                    min_distance = min_d
            pred_read_words.add(int(nearest_word_num))
        IoU = len(true_read_words.intersection(pred_read_words)) / len(true_read_words.union(pred_read_words))
        IoU_sum += IoU
        N += 1
    return IoU_sum / N

In [14]:
mIoU_stats(test_data_0, model)

  0%|          | 0/550 [00:00<?, ?it/s]

0.865041846517063

In [15]:
mIoU_stats(test_data_750, model)

  0%|          | 0/550 [00:00<?, ?it/s]

0.7355938741687935

In [16]:
mIoU_stats(test_data_1500, model)

  0%|          | 0/550 [00:00<?, ?it/s]

0.6613589766343705

In [17]:
mIoU_stats(test_data_3000, model)

  0%|          | 0/550 [00:00<?, ?it/s]

0.5592015631574512

# Visualization

In [21]:
example = test_data_3000[0]

In [22]:
model.eval()
with torch.no_grad(), autocast('cuda', enabled=(device.type=='cuda'), dtype=(torch.bfloat16 if (device.type=="cuda" and torch.cuda.is_bf16_supported()) else torch.float16)):
    page, mov_list, fix_list, _ = example
    page = page.to(device, dtype=torch.float16, non_blocking=True).unsqueeze(0)
    flow = model(page).squeeze(0)

In [23]:
warped_pts = warp_points_3d(mov_list, flow[0], flow[1])
wordbounds_tensor = page[0][1][0]

In [27]:
len(warped_pts[0])

72

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.patches import Rectangle

# --- helpers ---
def to_np(x):
    return x.detach().cpu().to(torch.float32).numpy() if torch.is_tensor(x) else np.asarray(x, dtype=np.float32)

def flow_to_T2HW(flow):
    f = to_np(flow)
    if f.ndim == 3 and f.shape[0] == 2:   # (2,H,W)
        return f[None, ...]
    if f.ndim == 4 and f.shape[1] == 2:   # (T,2,H,W)
        return f
    if f.ndim == 4 and f.shape[0] == 2:   # (2,T,H,W) -> (T,2,H,W)
        return np.transpose(f, (1,0,2,3))
    raise ValueError(f"Unexpected flow shape {f.shape}")

def bg_to_THW(bg, T, H, W):
    b = to_np(bg)
    if b.ndim == 2:              # (H,W) -> repeat over time
        assert b.shape == (H, W)
        return np.repeat(b[None, ...], T, axis=0)
    if b.ndim == 3:              # (T,H,W)
        assert b.shape == (T, H, W)
        return b
    raise ValueError(f"Unexpected background shape {b.shape}")

def y_to_disp(y, H):             # image y -> display y (Cartesian)
    return (H - 1) - np.asarray(y, dtype=np.float32)

# --- normalize inputs ---
flow_tchw = flow_to_T2HW(flow)        # (T,2,H,W)
T, _, H, W = flow_tchw.shape
dy_all = flow_tchw[:, 0]              # image coords: down-positive
dx_all = flow_tchw[:, 1]              # right-positive
bg_all  = bg_to_THW(wordbounds_tensor, T, H, W)

# global color scale across time
mag_full = np.hypot(dx_all, dy_all)
norm = Normalize(vmin=0.0, vmax=float(np.percentile(mag_full, 99)) + 1e-6, clip=True)

# quiver subsampling grid (image index space)
step = max(1, min(H, W) // 32)
Yi, Xi = np.mgrid[0:H:step, 0:W:step]

# traces
xs, ys = [], []
xs_true, ys_true = [], []
xs_warped, ys_warped = [], []

for p1, p2, p3 in zip(mov_list, fix_list, warped_pts[0]):
    # Unpack (t, x, y) for each; p3 may carry extra fields, we only need first three
    t1, x1, y1 = int(p1[0]), float(p1[1]), float(p1[2])
    t2, x2, y2 = int(p2[0]), float(p2[1]), float(p2[2])
    t3, x3, y3 = int(p3[0]), p3[1], p3[2]
    if torch.is_tensor(x3): x3 = float(to_np(x3))
    if torch.is_tensor(y3): y3 = float(to_np(y3))

    # choose time slice: prefer warped time, else moving time (clipped)
    t = t3 if 0 <= t3 < T else max(0, min(T - 1, t1))

    xs.append(x1); ys.append(y1)
    xs_true.append(x2); ys_true.append(y2)
    xs_warped.append(x3); ys_warped.append(y3)

    # time slice flow and background
    dx, dy = dx_all[t], dy_all[t]
    U = dx[::step, ::step]
    V = dy[::step, ::step]

    # convert to Cartesian display: flip background + coordinates + vertical vector
    bg_disp = np.flipud(bg_all[t])
    Y_disp  = (H - 1) - Yi
    V_disp  = -V

    ys_disp         = y_to_disp(ys, H)
    ys_true_disp    = y_to_disp(ys_true, H)
    ys_warped_disp  = y_to_disp(ys_warped, H)

    # --- draw figure (one "page") ---
    fig, axes = plt.subplots(1, 4, figsize=(30, 18), facecolor="white")

    # helper to show non-warp panels: white background, black graphics
    def show_bw(ax, x_pts, y_pts, title):
        ax.imshow(bg_disp, cmap='gray_r', origin='lower', alpha=1.0)  # white background with black features
        ax.plot(x_pts, y_pts, 'o-', color='red', markersize=3, linewidth=0.9)
        ax.set_title(title); ax.axis('off')

    show_bw(axes[0], xs,        ys_disp,        "Moving")
    show_bw(axes[1], xs_warped, ys_warped_disp, "Warped")
    show_bw(axes[2], xs_true,   ys_true_disp,   "Fixed")

    # --- ONLY CHANGED PANEL (4th image) ---
    # Darker background for better arrow contrast (robust scaling + dark cmap)
    bg = bg_disp.astype(np.float32)
    lo, hi = np.percentile(bg, 2), np.percentile(bg, 98)
    axes[3].imshow(bg, cmap='gray', origin='lower', vmin=lo, vmax=hi)  # darker than gray_r

    q = axes[3].quiver(
        Xi, Y_disp, U, V_disp, np.hypot(U, V),
        cmap='magma', norm=norm,
        angles='xy', scale_units='xy', scale=1.0,
        width=0.003, pivot='tail', linewidth=0.2
    )
    cb = plt.colorbar(q, ax=axes[3], fraction=0.046, pad=0.04)
    cb.set_label('Displacement (pixels)')
    axes[3].set_title("Warp field")

    # overlay the **last** source & warped points on the warp panel
    xs_last, ys_last_disp       = xs[-1],       ys_disp[-1]
    xs_warp_last, ys_warp_last  = xs_warped[-1], ys_warped_disp[-1]
    axes[3].plot([xs_last],      [ys_last_disp],      marker='o', markersize=5, color='red')
    axes[3].plot([xs_warp_last], [ys_warp_last],      marker='o', markersize=5, color='blue')

    axes[3].legend(loc='lower right', fontsize=8, frameon=False)

    plt.tight_layout()
    plt.show()

In [ ]:
xs = []
ys = []

xs_true = []
ys_true = []

xs_warped = []
ys_warped = []

flow_np = flow.detach().cpu().to(torch.float32).numpy()
warp_heat_map = np.linalg.norm(flow_np, axis=0)
vmin, vmax = warp_heat_map.min(), warp_heat_map.max()


for p1, p2, p3, whm in zip(mov_list, fix_list, warped_pts[0], warp_heat_map[:72]):
    t1, x1, y1 = p1
    t2, x2, y2 = p2
    t3, x3, y3 = p3

    xs.append(x1)
    ys.append(y1)

    xs_true.append(x2)
    ys_true.append(y2)

    xs_warped.append(x3.detach().cpu().to(torch.float32).numpy())
    ys_warped.append(y3.detach().cpu().to(torch.float32).numpy())
    
    _, axes = plt.subplots(1, 4, figsize=(15, 10))

    axes[0].plot(xs, ys, 'ro', linestyle="--", linewidth=1.5, markersize=3)
    axes[0].matshow(wordbounds_tensor.detach().cpu().numpy().squeeze(), cmap='Greys')

    axes[1].plot(xs_warped, ys_warped, 'ro', linestyle="--", linewidth=1.5, markersize=3)
    axes[1].matshow(wordbounds_tensor.detach().cpu().numpy().squeeze(), cmap='Greys')

    axes[2].plot(xs_true, ys_true, 'ro', linestyle="--", linewidth=1.5, markersize=3)
    axes[2].matshow(wordbounds_tensor.detach().cpu().numpy().squeeze(), cmap='Greys')

    im = axes[3].imshow(whm, cmap='magma', vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=axes[3], fraction=0.046, pad=0.04)
    axes[3].set_title("Warp magnitude")
    axes[3].axis('off')